In [46]:
import onnxruntime as ort 
import json
import os

In [ ]:
# ===== Creating a sample input of 10 data points ===== 

# Feature order
FEATURES = [
    "load_Rotary_C5",
    "position_Linear_X",
    "position_Linear_Z",
    "spindlespeed_actual_Rotary_C5",
    "pathfeedrate_Path_Path_1"
]

# 10 sequential samples (simple values for clarity)
sample_2d = [
    [2.0, 18.8, 10.1, 35.0, 239.2],
    [1.0, 18.8, 10.1, 35.0, 239.2],
    [1.0, 16.2, 3.9,  35.0, 239.2],
    [2.0, 16.2, 0.8,  35.0, 10.7],
    [1.0, 16.2, 0.8,  35.0, 10.7],
    [1.5, 15.0, 0.8,  40.0, 12.0],
    [1.8, 14.5, 1.2,  40.0, 15.0],
    [2.1, 14.0, 2.0,  45.0, 18.0],
    [2.4, 13.5, 3.5,  45.0, 20.0],
    [2.8, 13.0, 5.0,  50.0, 25.0],
]

print("2D Shape:", len(sample_2d), "x", len(sample_2d[0]))
print(sample_2d[:1])


2D Shape: 10 x 5
[[2.0, 18.8, 10.1, 35.0, 239.2]]


In [48]:
# ==== Get the min, max values per feature from the svaed scaler ==== 
# (This scaler may not be saved by the backend team)
# (They will need to save the min, max pairs per feature in their own fashion)

# I will only load this scaler (made during training) to fetch thise min, max values. 

import pickle
with open("../artifacts/latest/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

SCALER_MIN = scaler.data_min_.tolist()
SCALER_MAX = scaler.data_max_.tolist()

print("SCALER_MIN =", SCALER_MIN)
print("SCALER_MAX =", SCALER_MAX)

SCALER_MIN = [1.0, -0.0885, -5.6411, 13.0, 0.23]
SCALER_MAX = [52.0, 19.8566, 16.1201, 431.0, 241.0]


In [49]:
scaler.data_min_

array([ 1.    , -0.0885, -5.6411, 13.    ,  0.23  ])

In [50]:
scaler.data_max_

array([ 52.    ,  19.8566,  16.1201, 431.    , 241.    ])

In [51]:
def minmax_normalize_2d(data_2d, min_vals, max_vals):
    normalized = []

    for row in data_2d:
        new_row = []
        for i in range(len(row)):
            x = row[i]
            min_v = min_vals[i]
            max_v = max_vals[i]

            # Avoid divide by zero
            if max_v == min_v:
                value = 0.0
            else:
                value = (x - min_v) / (max_v - min_v)

            new_row.append(value)

        normalized.append(new_row)

    return normalized


normalized_2d = minmax_normalize_2d(sample_2d, SCALER_MIN, SCALER_MAX)
print("Normalized first row:")
print(normalized_2d[0])

Normalized first row:
[0.0196078431372549, 0.9470245824789046, 0.7233562487362828, 0.05263157894736842, 0.992523985546372]


In [56]:
# Verifying the normalization result by using the scaler itself to apply it: 
row = sample_2d[0]
sklearn_norm = scaler.transform([row])[0]

print("Manual :", normalized_2d[0])
print("Sklearn:", sklearn_norm.tolist())

Manual : [0.0196078431372549, 0.9470245824789046, 0.7233562487362828, 0.05263157894736842, 0.992523985546372]
Sklearn: [0.0196078431372549, 0.9470245824789046, 0.7233562487362828, 0.05263157894736842, 0.992523985546372]


c:\Users\kashika.p\AppData\Local\miniconda3\envs\ics_baseline_model\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [57]:
# Tried to replicate a one-by-one sample row append to create the final batch 
# Normlization can be applied similarly too. 

SEQUENCE_LENGTH = 10
buffer = []

def push_observation(row):
    buffer.append(row)

    if len(buffer) == SEQUENCE_LENGTH:
        return [buffer]  # ready for inference
    else:
        return None


for row in normalized_2d:
    input_for_model = push_observation(row)
    if input_for_model:
        print("Model ready input shape:",
              len(input_for_model), len(input_for_model[0]), len(input_for_model[0][0]))

Model ready input shape: 1 10 5


In [59]:
type(input_for_model)

list

In [67]:
input_for_model

[[[0.0196078431372549,
   0.9470245824789046,
   0.7233562487362828,
   0.05263157894736842,
   0.992523985546372],
  [0.0,
   0.9470245824789046,
   0.7233562487362828,
   0.05263157894736842,
   0.992523985546372],
  [0.0,
   0.8166667502293796,
   0.43844549013841144,
   0.05263157894736842,
   0.992523985546372],
  [0.0196078431372549,
   0.8166667502293796,
   0.29599011083947574,
   0.05263157894736842,
   0.043485484071935865],
  [0.0,
   0.8166667502293796,
   0.29599011083947574,
   0.05263157894736842,
   0.043485484071935865],
  [0.00980392156862745,
   0.7565015968834451,
   0.29599011083947574,
   0.0645933014354067,
   0.04888482784400049],
  [0.01568627450980392,
   0.7314327829893056,
   0.31437145010385453,
   0.0645933014354067,
   0.0613448519333804],
  [0.021568627450980395,
   0.7063639690951662,
   0.3511341286326121,
   0.07655502392344497,
   0.0738048760227603],
  [0.027450980392156862,
   0.6812951552010268,
   0.42006415087403265,
   0.07655502392344497,
   0

In [60]:
# ===== Run inference on model ======

# Start the onnx runtime session: 
session = ort.InferenceSession(
    "../artifacts/latest/models/model.onnx",
    providers=["CPUExecutionProvider"]
)

# get the input_name expected by onnx for input: 
input_name = session.get_inputs()[0].name  # this is "input_sequence" 

# Pass input to onnx session and run the prediction
pred_onnx = session.run(None,{input_name: input_for_model})[0]

pred_onnx.shape

(1, 10, 5)

In [61]:
session.run(None,{input_name: input_for_model})

[array([[[-0.01472788,  0.812611  ,  0.28045687,  0.05534304,
           0.1033814 ],
         [-0.02925825,  0.8027732 ,  0.31625313,  0.0543681 ,
           0.17088573],
         [-0.03257129,  0.80824363,  0.3393497 ,  0.05397487,
           0.21139368],
         [-0.02986941,  0.8171788 ,  0.34190118,  0.05321672,
           0.21675639],
         [-0.02327159,  0.82020533,  0.32279414,  0.05221587,
           0.1853884 ],
         [-0.01750533,  0.81728595,  0.29212093,  0.0516126 ,
           0.13211526],
         [-0.0119879 ,  0.81050795,  0.26218018,  0.05174937,
           0.07633369],
         [-0.00396273,  0.8032008 ,  0.240879  ,  0.05256348,
           0.03155933],
         [ 0.00814239,  0.79758936,  0.23009294,  0.05370995,
           0.00290111],
         [ 0.02347533,  0.794279  ,  0.22769552,  0.05476174,
          -0.01048966]]], dtype=float32)]

In [ ]:
pred_onnx[:,4,0] # this is the point to test for anomaly ----> Baseline to plot for the 5-th timestamp in the series of 10 points recieved. 

array([-0.02327159], dtype=float32)

In [62]:
pred_onnx

array([[[-0.01472788,  0.812611  ,  0.28045687,  0.05534304,
          0.1033814 ],
        [-0.02925825,  0.8027732 ,  0.31625313,  0.0543681 ,
          0.17088573],
        [-0.03257129,  0.80824363,  0.3393497 ,  0.05397487,
          0.21139368],
        [-0.02986941,  0.8171788 ,  0.34190118,  0.05321672,
          0.21675639],
        [-0.02327159,  0.82020533,  0.32279414,  0.05221587,
          0.1853884 ],
        [-0.01750533,  0.81728595,  0.29212093,  0.0516126 ,
          0.13211526],
        [-0.0119879 ,  0.81050795,  0.26218018,  0.05174937,
          0.07633369],
        [-0.00396273,  0.8032008 ,  0.240879  ,  0.05256348,
          0.03155933],
        [ 0.00814239,  0.79758936,  0.23009294,  0.05370995,
          0.00290111],
        [ 0.02347533,  0.794279  ,  0.22769552,  0.05476174,
         -0.01048966]]], dtype=float32)